# MARV on Llama-3.2-1B (Colab T4)

Extract a **vindex**, browse what the model knows, edit a feature constellation on
the live model and **measure what broke**. Also a real weight-space diff of
`Llama-3.2-1B` (base) vs `Llama-3.2-1B-Instruct`.

First cell is dominated by the model download (~2.5 GB). Runtime: **T4 GPU**.


In [ ]:
!pip install -q 'transformers>=4.45' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load + extract


In [ ]:
import torch, numpy as np, marv
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Ungated Llama-3.2-1B mirror. Alternatives:
#   meta-llama/Llama-3.2-1B-Instruct   (needs an HF token)
#   Qwen/Qwen2.5-1.5B-Instruct         (bigger, stronger recall, slower)
NAME = 'unsloth/Llama-3.2-1B-Instruct'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16).to(device).eval()
print(model.config.num_hidden_layers, 'layers  hidden', model.config.hidden_size,
      ' intermediate', model.config.intermediate_size, ' vocab', model.config.vocab_size)

vindex = marv.extract(model, model_name=NAME)
marv.build_down_meta(vindex, device=device)   # GPU: ~seconds even at 128k vocab
print('bands:', vindex.layer_bands)

## Browse: `describe_entity`

Bare-embedding gate-KNN + logit-lens. Fast, no forward pass. `feature -> promoted
tokens`, not a knowledge graph.


In [ ]:
for entity in ['France', 'Germany', 'Einstein']:
    print(f'\n=== {entity} ===')
    for r in marv.describe_entity(vindex, tok, entity, k_features=4):
        print('  ', r)

## Contextual constellation

The bare embedding is a weak query on most models. Pass the live `model` +
the actual fact prompt (differenced against a baseline) to query the real
hidden state -- much sharper.


In [ ]:
con = marv.constellation(vindex, tok, 'France', model=model,
                         prompt='The capital of France is',
                         baseline_prompt='The capital of',
                         per_layer=4, device=device)
for r in con[:12]:
    print(f'  L{r.layer:>2} f{r.feature:<5} sim={r.sim:.2f}  -> {r.tokens[:3]}')

## Edit + measure


In [ ]:
P = marv.Probe
battery = [
    P('The capital of France is', 'Paris', ('target',)),
    P('The French capital is', 'Paris', ('target',)),
    P('The capital of Italy is', 'Rome', ('neighbour',)),
    P('The capital of Germany is', 'Berlin', ('neighbour',)),
    P('The Eiffel Tower is in', 'Paris', ('neighbour',)),
    P('The capital of Japan is', 'Tokyo', ('control',)),
    P('Water is made of hydrogen and', 'oxygen', ('control',)),
    P('The opposite of hot is', 'cold', ('control',)),
]

feats = [(r.layer, r.feature) for r in con[:6]]
rep = marv.study_edit(model, tok, marv.suppress(model, feats), battery, device=device)
rep.show()

In [ ]:
rep.show(full=True)

## Weight-space diff: base vs. instruct

Same architecture (16 layers), so `marv.diff` compares feature by feature --
which neurons instruction-tuning moved, and whether it changed what they *fire
on* (`gate_cos`) or what they *promote* (`down_cos`).


In [ ]:
del model; import gc; gc.collect(); torch.cuda.empty_cache()
base_m = AutoModelForCausalLM.from_pretrained('unsloth/Llama-3.2-1B', torch_dtype=torch.float16)
vindex_base = marv.extract(base_m, model_name='unsloth/Llama-3.2-1B')
marv.build_down_meta(vindex_base, device=device)
del base_m; gc.collect()

deltas = marv.diff(vindex_base, vindex)
for d in marv.most_changed(deltas, k=12):
    b,_ = marv.describe_feature(vindex_base, d.layer, d.feature_idx, k=3)
    a,_ = marv.describe_feature(vindex,      d.layer, d.feature_idx, k=3)
    bw = [w.strip() for w in tok.batch_decode([[int(t)] for t in b])]
    aw = [w.strip() for w in tok.batch_decode([[int(t)] for t in a])]
    print(f'L{d.layer:>2} f{d.feature_idx:<5} gate_cos={d.gate_cos_sim:+.3f} down_cos={d.down_cos_sim:+.3f}  {bw} -> {aw}')

## Next
- `notebooks/marv_edit_eval_colab.ipynb` -- the full efficacy/specificity frontier.
- `marv.extract_streaming(dir)` for a checkpoint too big for T4 RAM.
